In [ ]:
## Environment setup

# All dependencies are listed in `requirements.txt`.

# Install environment before running the notebook:

# pip install -r requirements.txt
# python -m spacy download en_core_web_sm

In [ ]:
## Import libraries
# This section loads all required libraries used in the data processing pipeline.

from psycopg2 import connect
# Core data processing
import pandas as pd
from deep_translator import GoogleTranslator
# NLP processing
import spacy
# Database connection
import pysentiment2 as ps
import re

In [ ]:
## Database connection

cnx = connect(
    user="your_USER",
    password="your_PASSWORD",
    host="localhost",
    database="your_DATABASE"
)
cursor = cnx.cursor()

In [ ]:
# data processing pipeline
sql = "SELECT * FROM reviews"

cursor.execute(sql)

data = []
for row in cursor:
    data.append(list(row))


data = pd.DataFrame(data, columns=["id", "product", "productcode", "review"])

data


DATA PREPROCESSING

In [ ]:
nlp = spacy.load("en_core_web_sm")

In [ ]:
# Translate reviews texts to English - lexicons are in English
# Non-English reviews are translated into English to ensure consistent sentiment scoring.

translator = GoogleTranslator(source="cs", target="en")

data["review_en"] = data["review"].apply(
    lambda x: translator.translate(x) if isinstance(x, str) else x
)

data

In [ ]:
# Entire texts are lowercase

data["review_en"] = data["review_en"].str.lower()
data.head()

In [ ]:
# Removing numbers - no value for sentiment analysis

def remove_numbers(text):
    cleaned_text = re.sub(r'\b\d+(?:\.\d+)?\b', '', text)
    return cleaned_text
data["review_en"] = data["review_en"].apply(remove_numbers)
data.head()

In [ ]:
# Remove punctuation marks

for index, row in data.iterrows():
    sentence = nlp(row["review_en"])
    stop_word_free_text = " ".join(
        token.text for token in sentence if not token.is_punct)
    data.loc[index, "review_clean"] = stop_word_free_text

data.head()

In [ ]:
# Ensure we have only 1 space between each 2 words

def keep_only_one_blank_space(text):
    cleaned_text = re.sub(r'\s+', ' ', text).strip()
    return cleaned_text

data["review_en"] = data["review_en"].apply(keep_only_one_blank_space)

In [ ]:
# Ensure there are no multiple spaces between words and that only a single space is used.

print(str(data[data.index ==0]["review_en"]))

In [ ]:
# Lemmatisation

def lemmatize_text(text):
    doc = nlp(text)
    return " ".join(token.lemma_ for token in doc)

data["review_en"] = data["review_en"].apply(lemmatize_text)
data.head()

In [ ]:
# Removing stop words

# Stop word removing except for these that are needed in sentiment analysis

stop_words_important_for_sa = ["not", "no", "never", "nothing", "none", "cannot", "don\'t", "doesn\'t", "won\'t",
"isn\’t","very", "extremely", "so", "too", "quite", "really", "absolutely", "totally", "especially",
"incredibly", "super", "better", "best", "worse", "worst", "more", "less", "most", "least",
"can", "could", "should", "would", "will", "may", "might", "must", "shall", "but", "however",
"although", "yet", "unless", "absolutely", "definitely", "certainly", "just", "only", "simply",
"i", "we", "you", "me", "your", "good", "bad", "happy", "sad", "amazing", "horrible", "wonderful"]

for word in stop_words_important_for_sa:
    nlp.Defaults.stop_words.discard(word)

# Ensure all "article" entries are strings
data["review_en"] = data["review_en"].apply(lambda x: str(x) if not pd.isna(x) else "")

for index, row in data.iterrows():
    sentence = nlp(row["review_en"])
    stop_word_free_text = " ".join([token.text for token in sentence if not token.is_stop])
    data.loc[index, "review_en"] = stop_word_free_text

In [ ]:
hiv4 = ps.HIV4()
hiv4_lexicon = pd.read_csv(hiv4.PATH, sep=",")

hiv4_lexicon.loc[(hiv4_lexicon["Positiv"].isnull()) & (hiv4_lexicon["Negativ"].isnull()), "Neutral"] = "Neutral"
hiv4_lexicon.loc[hiv4_lexicon["Positiv"] == "Positiv", "Positiv_HIV4"] = 1
hiv4_lexicon.loc[hiv4_lexicon["Positiv"].isna(), "Positiv_HIV4"] = 0
hiv4_lexicon.loc[hiv4_lexicon["Negativ"] == "Negativ", "Negativ_HIV4"] = 1
hiv4_lexicon.loc[hiv4_lexicon["Negativ"].isna(), "Negativ_HIV4"] = 0
hiv4_lexicon.loc[hiv4_lexicon["Neutral"] == "Neutral", "Neutral_HIV4"] = 1
hiv4_lexicon.loc[hiv4_lexicon["Neutral"].isna(), "Neutral_HIV4"] = 0
hiv4_lexicon = hiv4_lexicon[["Entry", "Positiv_HIV4", "Negativ_HIV4", "Neutral_HIV4"]]
hiv4_lexicon = hiv4_lexicon.rename(columns={"Entry": "Word"})

lexicon = {
"positive": set(hiv4_lexicon[hiv4_lexicon['Positiv_HIV4'] == 1]['Word'].str.lower()),
"negative": set(hiv4_lexicon[hiv4_lexicon['Negativ_HIV4'] == 1]['Word'].str.lower()),
"neutral": set(hiv4_lexicon[hiv4_lexicon["Neutral_HIV4"] == 1]["Word"].str.lower())
}

In [ ]:
for sentiment, lex in lexicon.items():
        print(sentiment, lex)

SENTIMENT CALCULATION

In [ ]:
def calculate_sentiment_on_document(text):
    doc = nlp(text)
    sentiment_scores = {
        "positive": 0,
        "negative": 0,
        "neutral": 0
    }
# Iterate through document tokens and match them against
# the sentiment lexicon. When a match is found, increment
# the corresponding sentiment counter.

    for sentiment, lex in lexicon.items():
        print(sentiment, lex)

    token.text for token in doc

    return sentiment_scores


 # for word in doc:
  # word.text

In [ ]:
data["sentiment_modelled"] = data["review_en"].apply(calculate_sentiment_on_document)

In [ ]:
def calculate_sentiment_on_document(text):
    doc = nlp(text)
    sentiment_scores = {
        "positive": 0,
        "negative": 0,
        "neutral": 0
    }
    
# We iterate through both the sentiment lexicon and the tokens (words) in the processed document (doc = nlp(text)).
# If a token from the document is found in the lexicon, we determine its associated sentiment category
# (based on the lexicon key) and increment the corresponding sentiment counter in sentiment_scores.
    
    for token in doc:
        for sentiment, lex in lexicon.items():
            if token.text in lex:
                sentiment_scores[sentiment] += 1
    return sentiment_scores

In [ ]:
data["sentiment_modelled"] = data["review_en"].apply(calculate_sentiment_on_document)
data

In [ ]:
# Calculate scores and define sentiment categories
# New columns for sentiment scores

# (positive - negative) / (positive + negative + neutral + 1)

for index, row in data.iterrows():
    data.loc[index, "score"] = (
        (data["sentiment_modelled"][index]["positive"] - data["sentiment_modelled"][index]["negative"]) /
        (data["sentiment_modelled"][index]["positive"] + data["sentiment_modelled"][index]["negative"] +
        data["sentiment_modelled"][index]["neutral"] + 1)
    )


In [ ]:
data.score.unique()

In [ ]:
data.score ==0

In [ ]:
# score > 0.1 - positive
# score < 0.1 - negative
def assign_sentiment():
    pass

In [ ]:
# Apply function to each row
data["sentiment"] = data["score"].apply(assign_sentiment)